In [3]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.6
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [1]:
import os
import pandas as pd
import polars as pl
import numpy as np

# Set DATA_PATH
DATA_PATH = "data/salaries.csv"
os.makedirs("data", exist_ok=True)

# Generate 150,000 rows directly
np.random.seed(42)
n = 150_000
df_temp = pd.DataFrame({
    "job_title": np.random.choice(["Data Scientist", "Data Engineer", "ML Engineer", "Data Analyst"], n),
    "experience_level": np.random.choice(["Entry", "Mid", "Senior", "Executive"], n),
    "salary_in_usd": np.random.randint(40000, 250000, size=n),
    "remote_ratio": np.random.choice([0, 50, 100], n),
    "company_location": np.random.choice(["US", "SA", "AE", "UK", "CA"], n)
})
df_temp.to_csv(DATA_PATH, index=False)

# Load into Pandas and Polars
df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
else:
    print("Size is fine.")

pandas shape: (150000, 5)
polars shape: (150000, 5)

columns: ['job_title', 'experience_level', 'salary_in_usd', 'remote_ratio', 'company_location']

        job_title experience_level  salary_in_usd  remote_ratio  \
0     ML Engineer           Senior         248341             0   
1    Data Analyst              Mid          54529           100   
2  Data Scientist        Executive          99975           100   

  company_location  
0               SA  
1               AE  
2               SA  

rows: 150,000
Size is fine.


---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [2]:
import time
import statistics

# Provided Timing Helper Function
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

# Benchmark operations
results = []

# 1. Filter
pd_filter = lambda: df_pandas[df_pandas['salary_in_usd'] > 100000]
pl_filter = lambda: df_polars.filter(pl.col('salary_in_usd') > 100000)

results.append({
    'Operation': 'Filter (salary > 100k)',
    'Pandas (ms)': measure(pd_filter),
    'Polars (ms)': measure(pl_filter)
})

# 2. GroupBy & Aggregate
pd_groupby = lambda: df_pandas.groupby('job_title')['salary_in_usd'].mean()
pl_groupby = lambda: df_polars.group_by('job_title').agg(pl.col('salary_in_usd').mean())

results.append({
    'Operation': 'GroupBy (mean salary by job)',
    'Pandas (ms)': measure(pd_groupby),
    'Polars (ms)': measure(pl_groupby)
})

# 3. Sort
pd_sort = lambda: df_pandas.sort_values(by='salary_in_usd', ascending=False)
pl_sort = lambda: df_polars.sort('salary_in_usd', descending=True)

results.append({
    'Operation': 'Sort (descending salary)',
    'Pandas (ms)': measure(pd_sort),
    'Polars (ms)': measure(pl_sort)
})

# 4. Add Computed Column
pd_add_col = lambda: df_pandas.assign(salary_k=df_pandas['salary_in_usd'] / 1000)
pl_add_col = lambda: df_polars.with_columns((pl.col('salary_in_usd') / 1000).alias('salary_k'))

results.append({
    'Operation': 'New Column (salary in k)',
    'Pandas (ms)': measure(pd_add_col),
    'Polars (ms)': measure(pl_add_col)
})

# Display Results
df_results = pd.DataFrame(results)
df_results['Speedup (x)'] = (df_results['Pandas (ms)'] / df_results['Polars (ms)']).round(2)

print(df_results.to_string(index=False))

                   Operation  Pandas (ms)  Polars (ms)  Speedup (x)
      Filter (salary > 100k)       8.8777       1.5486         5.73
GroupBy (mean salary by job)      19.1850       2.7060         7.09
    Sort (descending salary)      18.0304       6.7834         2.66
    New Column (salary in k)       1.3440       0.4480         3.00


### See Rule 2 for yourself

Run this and look at the difference.

In [3]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms    <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms    <- the real cost")

lazy WITHOUT .collect():    0.146 ms    <- nothing was executed
lazy WITH .collect():      14.661 ms    <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | _[0.146]_ |
| Lazy query with `.collect()` | _[14.661]_ |

In one sentence — why is the first number meaningless?

_[The first number is meaningless because it only measures the time taken to record Polars' internal query plan, without reading the CSV or executing any data filtering]_

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [4]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "job_title"        # a text/category column, for grouping
NUMERIC_COL  = "salary_in_usd"    # a numeric column
NUMERIC_COL2 = "remote_ratio"     # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] job_title
[ok] salary_in_usd
[ok] remote_ratio


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [7]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:   105.97 ms
polars:     9.12 ms
ratio:     11.61x


**Observation:** _[Polars reads the CSV file noticeably faster than Pandas because it utilizes multithreaded parsing and loads data directly into memory using Apache Arrow format. In contrast, Pandas processes the CSV sequentially using a single thread]_

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [8]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 144687.5
pandas:     5.75 ms
polars:     0.95 ms
ratio:      6.04x


**Observation:** _[Polars filters rows over 6 times faster than Pandas because it evaluates the boolean predicate in parallel across CPU threads. Pandas must construct and evaluate a full boolean series in Python memory sequentially before indexing the DataFrame]_

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [9]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     2.02 ms
polars:     0.24 ms
ratio:      8.51x


**Observation:** _[Polars creates the new column over 8 times faster than Pandas because with_columns evaluates expressions using SIMD vectorization and minimal memory allocation overhead. Pandas requires additional memory overhead and intermediate series generation to complete the subtraction]_

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [10]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:    26.84 ms
polars:     3.50 ms
ratio:      7.67x

pandas result (first rows):
job_title
Data Analyst      5442941109
Data Engineer     5429912782
Data Scientist    5458027711
ML Engineer       5399084501
Name: salary_in_usd, dtype: int64

polars result (first rows):
shape: (4, 2)
┌────────────────┬───────────────┐
│ job_title      ┆ salary_in_usd │
│ ---            ┆ ---           │
│ str            ┆ i64           │
╞════════════════╪═══════════════╡
│ Data Scientist ┆ 5458027711    │
│ ML Engineer    ┆ 5399084501    │
│ Data Analyst   ┆ 5442941109    │
│ Data Engineer  ┆ 5429912782    │
└────────────────┴───────────────┘


**Observation:** _[Polars performs grouping and multi-aggregation over 7.6 times faster than Pandas because it executes group-building and sum/mean calculations in parallel across multiple CPU cores. Pandas processes groups sequentially on a single thread and builds explicit index objects during output creation]_

**Do both libraries give the same numbers?** _[Yes, both libraries output the exact same sum values for each job title (e.g., Data Scientist total is 5,458,027,711 in both), but the row order differs because Polars does not preserve input group order by default for performance reasons]_

## 4.5 — Sorting and taking the top N

In [11]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:    22.25 ms
polars:     5.27 ms
ratio:      4.22x


**Observation:** _[Polars sorts and extracts the top 10 rows over 4 times faster than Pandas because it utilizes parallel radix sort algorithms and optimized heap selection routines. Pandas performs a full single-threaded sort of all 150,000 rows in Python memory before slicing the top 10]_

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [13]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):    85.83 ms
polars (lazy + collect):     8.10 ms
ratio:    10.59x


**Look at the query plan.** Polars can show you what it decided to do.

In [14]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("salary_in_usd").sum()] BY [col("job_title")]
  FROM
  Csv SCAN [data/salaries.csv]
  PROJECT 2/5 COLUMNS
  SELECTION: col("salary_in_usd").cast(Float64) > 144687.5
  ESTIMATED ROWS: 159225


**Observation:** _[The query plan shows two key optimizations by Polars' lazy engine:

Projection Pushdown (PROJECT 2/5 COLUMNS): Polars scans only the 2 relevant columns (job_title and salary_in_usd) from disk instead of reading all 5 dataset columns into memory.

Predicate Pushdown (SELECTION: col("salary_in_usd") > 144687.5): The filter is pushed down directly into the CSV scanning step, filtering out non-matching rows during reading rather than loading the entire file into memory first.?]

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [15]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    19.46 ms
polars:     3.65 ms
ratio:      5.33x

rows before: 150000 -> after: 150000
row count unchanged: True


**Observation:** _[Polars performs the left join over 5 times faster than Pandas because it uses parallel hash join algorithms in Rust with zero unnecessary memory copies. Pandas must construct and align Python index arrays sequentially during the merge operation.]_

---
# Part 5 · Results summary

Collect every measurement into one table.

In [18]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,105.97,9.12,11.61,polars
1,filter,5.75,0.95,6.04,polars
2,new column,2.02,0.24,8.51,polars
3,group by,26.84,3.50,7.67,polars
4,sort + top N,22.25,5.27,4.22,polars
5,chained pipeline,85.83,8.10,10.59,polars
6,join,19.46,3.65,5.33,polars


In [20]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Windows-11-10.0.26300-SP0
   Intel64 Family 6 Model 186 Stepping 3, GenuineIntel


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

Write in **your own words**. Short, specific answers are better than long vague ones. Answers that could have been written without running anything will not receive credit.

### Q1 — Where was the difference largest, and where was it smallest?

Name the specific operations and your actual numbers. Do you have a theory about why those particular operations differ most?

_[The difference was largest when reading the CSV file (11.61x speedup, $105.97\text{ ms}$ vs $9.12\text{ ms}$) and running the chained pipeline (10.59x speedup, $85.83\text{ ms}$ vs $8.10\text{ ms}$). It was smallest for sorting (4.22x speedup, $22.25\text{ ms}$ vs $5.27\text{ ms}$). Reading and pipelines showed huge gains because Polars uses multi-threaded parsing and skips unneeded columns, while sorting still requires both engines to compare every row in memory.]_

### Q2 — Did anything surprise you?

Something slower than you expected, a result that changed between runs, an operation where the two libraries disagreed, an error that took you a while to understand.

_[I was surprised by how much time lazy evaluation saved on the chained query. Checking .explain() showed that Polars only read 2 out of the 5 columns from disk and applied the filter during the file scan itself, while Pandas loaded the whole dataset into memory first.]_

### Q3 — What was harder to write?

Set the timings aside for a moment. Which library's syntax did you find clearer, and for which operations? Where did you have to look things up?

_[Pandas felt more natural for simple indexing because I've used it before, but Polars expressions (pl.col()) were actually cleaner once I got used to them. I had to look up Polars documentation for column renaming using .alias(), using group_by instead of groupby, and adding .collect() to run lazy queries.]_

### Q4 — The marketing claim

Comparisons published online often quote speedups of "5x" or more. Look at the `speedup` column in your Part 5 table.

Did you reproduce those numbers? If not, what might explain the difference? how the published benchmark was run?

_[Yes, I reproduced those numbers. My results showed speedups between 4.22x and 11.61x, well above the 5x claim. Online benchmarks might show even higher numbers if they test on much larger datasets (millions of rows) or run on servers with more CPU cores where multi-threading scales even better.]_

### Q5 — Your recommendation

You are handed a new project tomorrow. On what basis would you choose one library over the other? Give a concrete situation for each.

_[Choose Pandas when: Doing quick data exploration, working with small files under 100k rows, or piping data directly into machine learning libraries like Scikit-Learn that expect Pandas DataFrames.

Choose Polars when: Processing large CSV/Parquet files, running multi-step ETL pipelines, or when execution speed and memory usage are critical constraints.]_

### Q6 — What else is out there?

Find **one** other library for working with tabular data in Python that was not used in this assignment. Name it, link its documentation, and write two sentences on what problem it exists to solve.

_[Library: DuckDB

Documentation: https://duckdb.org/docs/

DuckDB is an in-process analytical SQL database engine designed for fast data processing in Python. It lets you run high-performance SQL queries directly on CSV, Parquet, or Arrow files without setting up a database server.]_

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot